### Data Class

In [1]:
import random

class Sentiment:
    NEGATIVE = "NEGATIVE"
    NEUTRAL = "NEUTRAL"
    POSITIVE = "POSITIVE"

class Review:
    def __init__(self, text, score):
        self.text = text
        self.score = score
        self.sentiment = self.get_sentiment()
        
    def get_sentiment(self):
        if self.score <= 2:
            return Sentiment.NEGATIVE
        elif self.score == 3:
            return Sentiment.NEUTRAL
        else: #Score of 4 or 5
            return Sentiment.POSITIVE

class ReviewContainer:
    def __init__(self, reviews):
        self.reviews = reviews
        
    def get_text(self):
        return [x.text for x in self.reviews]
    
    def get_sentiment(self):
        return [x.sentiment for x in self.reviews]

    # Balances the dataset by undersampling the positive class to match the number of negative reviews
    def evenly_distribute(self):
        negative = list(filter(lambda x: x.sentiment == Sentiment.NEGATIVE, self.reviews))
        positive = list(filter(lambda x: x.sentiment == Sentiment.POSITIVE, self.reviews))
        # slices positive down to len(negative)
        positive_shrunk = positive[:len(negative)]
        self.reviews = negative + positive_shrunk
        # call this in place so positive and negative samples aren't grouped in blocks. Make sure that import random in imported
        random.shuffle(self.reviews)
        
        

### Load Data

In [2]:
import json

file_name = './data/sentiment/books_small_10000.json'

reviews = []
with open(file_name) as f:
    for line in f:
        review = json.loads(line)
        reviews.append(Review(review['reviewText'], review['overall']))
        
reviews[5].text
        

'I hoped for Mia to have some peace in this book, but her story is so real and raw.  Broken World was so touching and emotional because you go from Mia\'s trauma to her trying to cope.  I love the way the story displays how there is no "just bouncing back" from being sexually assaulted.  Mia showed us how those demons come for you every day and how sometimes they best you. I was so in the moment with Broken World and hurt with Mia because she was surrounded by people but so alone and I understood her feelings.  I found myself wishing I could give her some of my courage and strength or even just to be there for her.  Thank you Lizzy for putting a great character\'s voice on a strong subject and making it so that other peoples story may be heard through Mia\'s.'

### Prep Data

In [4]:
from sklearn.model_selection import train_test_split

# 33% is for testing, 66% is for training
training, test = train_test_split(reviews, test_size=0.33, random_state=42)

train_container = ReviewContainer(training)

test_container = ReviewContainer(test)


In [9]:
print(len(training))
print(len(test))

6700
3300


In [5]:
train_x = [x.text for x in training]
train_y = [x.sentiment for x in training]
print(train_x[0])
print(train_y[0])

test_x = [x.text for x in test]
test_y = [x.sentiment for x in test]
print(test_x[0])
print(test_y[0])

Olivia Hampton arrives at the Dunraven family home as cataloger of their extensive library. What she doesn't expect is a broken carriage wheel on the way. Nor a young girl whose mind is clearly gone, an old man in need of care himself (and doesn&#8217;t quite seem all there in Olivia&#8217;s opinion). Furthermore, Marion Dunraven, the only sane one of the bunch and the one Olivia is inexplicable drawn to, seems captive to everyone in the dusty old house. More importantly, she doesn't expect to fall in love with Dunraven's daughter Marion.Can Olivia truly believe the stories of sadness and death that surround the house, or are they all just local neighborhood rumor?Was that carriage trouble just a coincidence or a supernatural sign to stay away? If she remains, will the Castle&#8217;s dark shadows take Olivia down with them or will she and Marion long enough to declare their love?Patty G. Henderson has created an atmospheric and intriguing story in her Gothic tale. I found this to be an

In [6]:
train_container.evenly_distribute()
train_x = train_container.get_text()
train_y = train_container.get_sentiment()

# Balancing test_container makes test evaluation treat both classes with equal sample counts
# (which is common in tutorials/assignments to inspect balanced accuracy/F1), 
# though in real-world deployment you usually leave test data in its natural distribution.
test_container.evenly_distribute()
test_x = test_container.get_text()
test_y = test_container.get_sentiment()

print(train_y.count(Sentiment.POSITIVE))
print(train_y.count(Sentiment.NEGATIVE))

436
436


#### Bag of words vectorization

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
train_x_vectors = vectorizer.fit_transform(train_x)
# The above line is equal to:
# vectorizer.fit(train_x)
# train_x_vectors = vectorizer.transform(train_x)

print(train_x[0])
print(train_x_vectors[0])
print(train_x_vectors[0].toarray())

I've LOVED the entire Stephanie Plum series but having to wait forever for this book left me disappointed with the results.  I've grown tired of the same old story with Stephanie not being able to decide between Joe and Ranger.  Pick one and get one with it already!
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 39 stored elements and shape (1, 8906)>
  Coords	Values
  (0, 8467)	2
  (0, 4784)	1
  (0, 7929)	3
  (0, 2718)	1
  (0, 7490)	2
  (0, 5948)	1
  (0, 7002)	1
  (0, 1168)	1
  (0, 3671)	1
  (0, 8052)	2
  (0, 8572)	1
  (0, 3191)	1
  (0, 3177)	1
  (0, 7976)	1
  (0, 991)	1
  (0, 4616)	1
  (0, 4982)	1
  (0, 2284)	1
  (0, 8760)	3
  (0, 6624)	1
  (0, 3538)	1
  (0, 8046)	1
  (0, 5478)	1
  (0, 6844)	1
  (0, 5502)	1
  (0, 7533)	1
  (0, 5408)	1
  (0, 825)	1
  (0, 144)	1
  (0, 2041)	1
  (0, 874)	1
  (0, 4361)	1
  (0, 423)	2
  (0, 6358)	1
  (0, 5860)	1
  (0, 5514)	2
  (0, 3374)	1
  (0, 4277)	1
  (0, 375)	1
[[0 0 0 ... 0 0 0]]


In [8]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# This book is great !
# This book was so bad
# -> 'This', 'is', 'book' is not so important, 'great' and 'bad' is much more important
# TF-IDF: TF(Term Frequency), IDF(Inverse Document Frequency) - how rare or common a word is across all documents in the training set
# Very common words like 'the', 'is', or 'book' appear in almost every document, so there IDF is low - reducing their weight
# In scikit-learn, TfidfVectorizer applies smoothing and L2 row normalization by default so every document vector has unit length
# Unit length: the L2 norm(Euclidean length) of a document's feature vector equals 1
# Scaling every row to length 1 shifts focus to the relative proportions/importance of words rather than sheer review length

vectorizer = TfidfVectorizer()
train_x_vectors = vectorizer.fit_transform(train_x)

# Never use .fit() on test_x because the vectorizer learns vocabulary words, word frequencies, or IDF weights from the test reviews.
# That leaks information from the test set into your feature representation -> no longer a fair measure
# A trained classifier expects input vectors with the exact same feature columns (vocabularies) it saw during training -> fitting a vectorizer separately on test_x creates a different vocabulary (some training words may be missing, new test words may appear in different positions, and the total number of columns can change)
# This book is great -> {"this": 0, "book": 1, "is": 2, "great": 3, "was": 4, "bad": 5} (this whole dictionary is the vocabulary)
 
test_x_vectors = vectorizer.transform(test_x)

print(train_x[0])
print(train_x_vectors[0].toarray())


I've LOVED the entire Stephanie Plum series but having to wait forever for this book left me disappointed with the results.  I've grown tired of the same old story with Stephanie not being able to decide between Joe and Ranger.  Pick one and get one with it already!
[[0. 0. 0. ... 0. 0. 0.]]


## Classification

### Linear SVM



















In [9]:
from sklearn import svm

clf_svm = svm.SVC(kernel='linear')

clf_svm.fit(train_x_vectors, train_y)

clf_svm.predict(test_x_vectors[0])

array(['NEGATIVE'], dtype='<U8')

### Decision Tree

In [10]:
from sklearn.tree import DecisionTreeClassifier

clf_dec = DecisionTreeClassifier()
clf_dec.fit(train_x_vectors, train_y)

clf_dec.predict(test_x_vectors[0])


array(['NEGATIVE'], dtype='<U8')

### Naive Bayes

In [11]:
from sklearn.naive_bayes import GaussianNB

clf_gnb = GaussianNB()
#fit(X, y, sample_weight = None) - X: array
clf_gnb.fit(train_x_vectors.toarray(), train_y)

clf_gnb.predict(test_x_vectors[0].toarray())


array(['NEGATIVE'], dtype='<U8')

### Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression

clf_log = LogisticRegression()
clf_log.fit(train_x_vectors, train_y)

clf_log.predict(test_x_vectors[0])



array(['NEGATIVE'], dtype='<U8')

### Random Forest

In [13]:
from sklearn.ensemble import RandomForestClassifier

clf_rf = RandomForestClassifier()
clf_rf.fit(train_x_vectors, train_y)

clf_rf.predict(test_x_vectors[0])

array(['NEGATIVE'], dtype='<U8')

## Evaluation

In [ ]:
# Mean Accuracy on the give test data and labels
print(clf_svm.score(test_x_vectors, test_y))
print(clf_dec.score(test_x_vectors, test_y))
print(clf_gnb.score(test_x_vectors.toarray(), test_y))
print(clf_log.score(test_x_vectors, test_y))
print(clf_rf.score(test_x_vectors, test_y))

# -> When you compare models on the test set and pick the winner, the "information" leaking isn't the test data going into .fit()—
# it's using test scores (test_y) as a decision rule to choose which model gets lucky on that specific sample's random quirks.
# -> use cross-validation on the training set (GridSearchCV) to pick models and hyperparameters, and touch the test set only once at the end for your final score. 

0.8076923076923077
0.6418269230769231
0.6610576923076923
0.8028846153846154
0.78125


In [15]:
# F1 Scores: can be interpreted as a weighted average of the precision and recall, where an F1 score reaches its best value at 1 and worst score at 0.
# precision: out of all samples predicted as positive, how many are actually positive?
# recall: out of all actual positive samples, how many did the model catch
from sklearn.metrics import f1_score

f1_score(test_y, clf_svm.predict(test_x_vectors), average=None, labels=[Sentiment.POSITIVE, Sentiment.NEGATIVE])
#f1_score(test_y, clf_log.predict(test_x_vectors), average=None, labels=[Sentiment.POSITIVE, Sentiment.NEUTRAL, Sentiment.NEGATIVE])

# ex: [0.92, 0.093, 0.1] -> predict positive very good but neutral and negative are trash

array([0.80582524, 0.80952381])

In [16]:
print(train_y[0:5]) # there are many positives
print(train_y.count(Sentiment.POSITIVE)) # to see how many positives there are -> heavily biased
print(train_y.count(Sentiment.NEGATIVE))

['NEGATIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'NEGATIVE']
436
436


In [17]:
test_set = ['very fun', "bad book do not buy", 'horrible waste of time']
new_test = vectorizer.transform(test_set)

clf_svm.predict(new_test)


array(['POSITIVE', 'NEGATIVE', 'NEGATIVE'], dtype='<U8')

### Tuning our model (with Grid Search)

In [20]:
# default model settings rarely fit your specific dataset best—and guessing hyperparameters on your test set breaks honest evaluation
# GridSearchCV gives you a systematic, leak-free way to pick settings that generalize well to unseen data

# Run hyperparameter tuning with cross-validation to find the best combination of kernel and C for a Support Vector Machine (SVM) classifier on your training data.
from sklearn.model_selection import GridSearchCV

# kernel: test a linear decision boundary ('linear')
# vs a radial basis function boundary ('rbf), which can capture non-linear patterns
# C: controls the regularization penalty for misclassifying training points. 
# Higher C means fewer training errors allowed
parameters = {'kernel': ('linear', 'rbf'), 'C': (1,4,8,16,32)}

svc = svm.SVC()

# cv = 5 uses 5-fold cross-validation
# train_x_vectors and train_y are split into 5 folds;
# for each of the 10 parameter combinations, the model trains on 4 folds and validates on the remaining 1 fold, rotating through all 5 folds.
clf = GridSearchCV(svc, parameters, cv=5)
clf.fit(train_x_vectors, train_y)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': (1, ...), 'kernel': ('linear', ...)}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues

In [21]:
print(clf.score(test_x_vectors, test_y))

0.8076923076923077


In [24]:
for rank, score, params in sorted(
    zip(
        clf.cv_results_['rank_test_score'],
        clf.cv_results_['mean_test_score'],
        clf.cv_results_['params'],
    ),
    key=lambda x: (x[0], -x[1]),
):
    print(f"Rank {rank}: score={score:.4f} | params={params}")

Rank 1: score=0.8326 | params={'C': 1, 'kernel': 'linear'}
Rank 2: score=0.8199 | params={'C': 4, 'kernel': 'rbf'}
Rank 2: score=0.8199 | params={'C': 8, 'kernel': 'rbf'}
Rank 2: score=0.8199 | params={'C': 16, 'kernel': 'rbf'}
Rank 2: score=0.8199 | params={'C': 32, 'kernel': 'rbf'}
Rank 6: score=0.8188 | params={'C': 8, 'kernel': 'linear'}
Rank 6: score=0.8188 | params={'C': 16, 'kernel': 'linear'}
Rank 6: score=0.8188 | params={'C': 32, 'kernel': 'linear'}
Rank 9: score=0.8188 | params={'C': 1, 'kernel': 'rbf'}
Rank 10: score=0.8165 | params={'C': 4, 'kernel': 'linear'}


In [ ]:
# good! good -strip out the exclamation mark

## Saving Model

#### Save model

In [25]:
import pickle

with open('./models/sentiment_classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

#### Load model

In [27]:
with open('./models/sentiment_classifier.pkl', 'rb') as f:
    loaded_clf = pickle.load(f)

In [28]:
print(test_x[0])

loaded_clf.predict(test_x_vectors[0])

For me this book was just two and a half stars. The characters took too long to get together with silly misunderstandings. When they finally got together she runs away from him with another misunderstanding. The end felt too rushed. I was annoyed with the second book when they were little together because of the tour. I thought they would be together more in the last book and got disappointed when that didn't happened.


array(['NEGATIVE'], dtype='<U8')

In [ ]:
from sklearn.linear_model import Perceptron

clf = Perceptron(tol=1e-3, random_state=0)
clf.fit(X, y)  